# RAG Agentic AI Platform - Kaggle Research Pipeline

This notebook demonstrates the end-to-end RAG pipeline running entirely locally on a Kaggle T4/P100 instance.

## 1. Setup Environment

In [ ]:
!pip install -r ../requirements-kaggle.txt

## 2. Initialize Models

We load `Qwen/Qwen2.5-7B-Instruct` (or `Qwen/Qwen3-8B`) in 4-bit quantization, and `BAAI/bge-m3` directly onto the GPU.

In [ ]:
import os
import sys
sys.path.append('..')

from src.llm.hf_client import HFClient
from src.embeddings.bge_m3 import BGEM3Embedder
from src.reranking.bge_reranker import BGEReranker

print("Loading LLM in 4-bit...")
llm = HFClient(model="Qwen/Qwen2.5-7B-Instruct")

print("Loading Embedder...")
embedder = BGEM3Embedder(device="cuda")

print("Loading Reranker...")
reranker = BGEReranker(device="cuda")


## 3. Initialize Local Qdrant and Index Documents

In [ ]:
from src.indexing.qdrant_store import QdrantStore
from config.settings import QdrantConfig
from src.core.models import Chunk, DocumentMetadata
from datetime import datetime

qdrant = QdrantStore(config=QdrantConfig(mode="disk", collection_name="kaggle_research"))
qdrant.create_collection(dense_dim=1024, recreate=True)

sample_chunks = [
    Chunk(chunk_id="1", document_id="doc1", content="Machine learning is a field of inquiry devoted to understanding and building methods that 'learn'.", chunk_index=0, token_count=20, section_title="Intro", section_hierarchy=[], metadata=DocumentMetadata(source="wiki", date=datetime.now(), department="science", version="1", tags=["ml"], access_level="public")),
    Chunk(chunk_id="2", document_id="doc2", content="Retrieval-Augmented Generation (RAG) is a technique for enhancing the accuracy and reliability of generative AI models with facts fetched from external sources.", chunk_index=0, token_count=25, section_title="Intro", section_hierarchy=[], metadata=DocumentMetadata(source="wiki", date=datetime.now(), department="science", version="1", tags=["rag"], access_level="public"))
]

# Create dense and sparse embeddings
texts = [c.content for c in sample_chunks]
dense_vecs, sparse_vecs = embedder.embed_dense_and_sparse(texts)

# Upsert
qdrant.upsert_chunks(sample_chunks, dense_vectors=dense_vecs, sparse_vectors=sparse_vecs)
print("Indexed sample documents!")

## 4. Run Retrieval and Reranking

In [ ]:
query = "What is RAG?"
query_dense = embedder.embed_query(query)

# Dense Search
retrieved = qdrant.search_dense(query_dense, top_k=5)
print("Initial Retrieval:")
for r in retrieved:
    print(f"- {r.chunk.content} (Score: {r.score})")

# Rerank
reranked = reranker.rerank(query, retrieved, top_k=2)
print("\nAfter Reranking:")
for r in reranked:
    print(f"- {r.chunk.content} (Score: {r.score})")

## 5. Generate Answer via LLM

In [ ]:
context = "\n".join([r.chunk.content for r in reranked])
prompt = f"Use the following context to answer the question.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"

answer = llm.generate(prompt, max_tokens=100)
print("\nLLM Answer:\n", answer)